In [1]:
!pip install PyPDF2
!pip install rich ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 26.5 MB/s eta 0:00:00


In [2]:
#Define pdf as target file and load model llama 3.2-1B, which is cheap to use, since this process does not need the best model
pdf_path = 'target.pdf'
DEFAULT_MODEL = "meta-llama/Llama-3.2-1B-Instruct"

In [3]:
#Imports
import PyPDF2
from typing import Optional
import os
import torch
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer

from tqdm.notebook import tqdm
import warnings

warnings.filterwarnings('ignore')

In [4]:
#Make sure file exists
def validate_pdf(file_path: str) -> bool:
    if not os.path.exists(file_path):
        print(f"Error: File not found at path: {file_path}")
        return False
    if not file_path.lower().endswith('.pdf'):
        print("Error: File is not a PDF")
        return False
    return True

In [5]:
#Definition of function that splits the input text into chunks and tells if the text exceeds the set limit of 100.000 characters and where it splits.
def extract_text_from_pdf(file_path: str, max_chars: int = 100000) -> Optional[str]:
    if not validate_pdf(file_path):
        return None

    try:
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)

            num_pages = len(pdf_reader.pages)
            print(f"Processing PDF with {num_pages} pages...")

            extracted_text = []
            total_chars = 0

            for page_num in range(num_pages):
                page = pdf_reader.pages[page_num]
                text = page.extract_text()

                if total_chars + len(text) > max_chars:
                    remaining_chars = max_chars - total_chars
                    extracted_text.append(text[:remaining_chars])
                    print(f"Reached {max_chars} character limit at page {page_num + 1}")
                    break

                extracted_text.append(text)
                total_chars += len(text)
                print(f"Processed page {page_num + 1}/{num_pages}")

            final_text = '\n'.join(extracted_text)
            print(f"\nExtraction complete! Total characters: {len(final_text)}")
            return final_text

    except PyPDF2.PdfReadError:
        print("Error: Invalid or corrupted PDF file")
        return None
    except Exception as e:
        print(f"An unexpected error occurred: {str(e)}")
        return None


In [6]:
#Load pdf metadata
def get_pdf_metadata(file_path: str) -> Optional[dict]:
    if not validate_pdf(file_path):
        return None

    try:
        with open(file_path, 'rb') as file:
            pdf_reader = PyPDF2.PdfReader(file)
            metadata = {
                'num_pages': len(pdf_reader.pages),
                'metadata': pdf_reader.metadata
            }
            return metadata
    except Exception as e:
        print(f"Error extracting metadata: {str(e)}")
        return None

In [7]:
#Print metadata
print("Extracting metadata...")
metadata = get_pdf_metadata(pdf_path)
if metadata:
    print("\nPDF Metadata:")
    print(f"Number of pages: {metadata['num_pages']}")
    print("Document info:")
    for key, value in metadata['metadata'].items():
        print(f"{key}: {value}")

#Starts the extraction and displays portion as preview
print("\nExtracting text...")
extracted_text = extract_text_from_pdf(pdf_path)

if extracted_text:
    print("\nPreview of extracted text (first 500 characters):")
    print("-" * 50)
    print(extracted_text[:500])
    print("-" * 50)
    print(f"\nTotal characters extracted: {len(extracted_text)}")

#Saves file in case inspection is needed
if extracted_text:
    output_file = 'extracted_text.txt'
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(extracted_text)
    print(f"\nExtracted text has been saved to {output_file}")

Extracting metadata...

PDF Metadata:
Number of pages: 13
Document info:
/Author: Hanne Ingmer
/CreationDate: D:20171005155632+05'30'
/Creator: LaTeX with hyperref package
/Keywords: Staphylococcus aureus, Staphylococcus schleiferi, quorum sensing, agr, quorum sensing inhibition, auto-inducing peptide, cross-talk, anti-virulence therapy
/ModDate: D:20250221180541+01'00'
/PTEX.Fullbanner: This is MiKTeX-pdfTeX 2.9.5496 (1.40.15)
/Producer: pdfTeX-1.40.15
/Subject: Staphylococci are associated with both humans and animals.
/Title: Cross-Talk between Staphylococcus aureus and Other Staphylococcal Species via the agr Quorum Sensing System
/Trapped: /False

Extracting text...
Processing PDF with 13 pages...
Processed page 1/13
Processed page 2/13
Processed page 3/13
Processed page 4/13
Processed page 5/13
Processed page 6/13
Processed page 7/13
Processed page 8/13
Processed page 9/13
Processed page 10/13
Processed page 11/13
Processed page 12/13
Processed page 13/13

Extraction complete! To

In [8]:
#Use GPU
device = "cuda" if torch.cuda.is_available() else "cpu"

#Prompt to instruct LLM to extract only the most relevant contents of the file which are needed for a podcast about the content.
SYS_PROMPT = """
You are a world class text pre-processor, here is the raw data from a PDF, please parse and return it in a way that is crispy and usable to send to a podcast writer.

The raw data is messed up with new lines, Latex math and you will see fluff that we can remove completely. Basically take away any details that you think might be useless in a podcast author's transcript.

Remember, the podcast could be on any topic whatsoever so the issues listed above are not exhaustive

Please be smart with what you remove and be creative ok?

Remember DO NOT START SUMMARIZING THIS, YOU ARE ONLY CLEANING UP THE TEXT AND RE-WRITING WHEN NEEDED

Be very smart and aggressive with removing details, you will get a running portion of the text and keep returning the processed text.

PLEASE DO NOT ADD MARKDOWN FORMATTING, STOP ADDING SPECIAL CHARACTERS THAT MARKDOWN CAPATILISATION ETC LIKES

ALWAYS start your response directly with processed text and NO ACKNOWLEDGEMENTS about my questions ok?
Here is the text:
"""

In [9]:
#Function to split the text into chunks for the model to easier parse the content
def create_word_bounded_chunks(text, target_chunk_size):
    """
    Split text into chunks at word boundaries close to the target chunk size.
    """
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        word_length = len(word) + 1
        if current_length + word_length > target_chunk_size and current_chunk:
            chunks.append(' '.join(current_chunk))
            current_chunk = [word]
            current_length = word_length
        else:
            current_chunk.append(word)
            current_length += word_length

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return chunks

In [10]:
#Load transformers and models from huggingface using token saved in colab
accelerator = Accelerator()
model = AutoModelForCausalLM.from_pretrained(
    DEFAULT_MODEL,
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
    device_map=device,
)
tokenizer = AutoTokenizer.from_pretrained(DEFAULT_MODEL, use_safetensors=True)
model, tokenizer = accelerator.prepare(model, tokenizer)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

In [11]:
#Instruct model to process a chunk at a time
def process_chunk(text_chunk, chunk_num):
    """Process a chunk of text and return both input and output for verification"""
    conversation = [
        {"role": "system", "content": SYS_PROMPT},
        {"role": "user", "content": text_chunk},
    ]

    prompt = tokenizer.apply_chat_template(conversation, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            temperature=0.7,
            top_p=0.9,
            max_new_tokens=512
        )

    processed_text = tokenizer.decode(output[0], skip_special_tokens=True)[len(prompt):].strip()


    #Print for inspection
    print(f"INPUT TEXT:\n{text_chunk[:500]}...")
    print(f"\nPROCESSED TEXT:\n{processed_text[:500]}...")
    print(f"{'='*90}\n")

    return processed_text

In [12]:
#Run the functions
INPUT_FILE = "extracted_text.txt"
CHUNK_SIZE = 1000

chunks = create_word_bounded_chunks(extracted_text, CHUNK_SIZE)
num_chunks = len(chunks)


In [13]:
#Check number of chunks
num_chunks

62

In [14]:
#Read the file with number of chunks and save as cleaned version of it.
with open(INPUT_FILE, 'r', encoding='utf-8') as file:
    text = file.read()

num_chunks = (len(text) + CHUNK_SIZE - 1) // CHUNK_SIZE

output_file = f"clean_{os.path.basename(INPUT_FILE)}"

In [15]:
#Process text
processed_text = ""

with open(output_file, 'w', encoding='utf-8') as out_file:
    for chunk_num, chunk in enumerate(tqdm(chunks, desc="Processing chunks")):
        processed_chunk = process_chunk(chunk, chunk_num)
        processed_text += processed_chunk + "\n"

        out_file.write(processed_chunk + "\n")
        out_file.flush()

Processing chunks:   0%|          | 0/62 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


INPUT TEXT:
fmicb-07-01733 October 5, 2017 Time: 15:56 # 1 ORIGINAL RESEARCH published: 08 November 2016 doi: 10.3389/fmicb.2016.01733 Edited by: Susanne Fetzner, University of Münster, Germany Reviewed by: Mattias Collin, Lund University, Sweden Shinya Watanabe, Jichi Medical University, Japan *Correspondence: Hanne Ingmer hi@sund.ku.dk †These authors have contributed equally to this work. Specialty section: This article was submitted to Infectious Diseases, a section of the journal Frontiers in Microbiolo...

PROCESSED TEXT:
: 08 November 2016 doi: 10.3389/fmicb.2016.01733 Edited by: Susanne Fetzner, University of Münster, Germany Reviewed by: Mattias Collin, Lund University, Sweden Shinya Watanabe, Jichi Medical University, Japan 
Correspondence: Hanne Ingmer hi@sund.ku.dk †These authors have contributed equally to this work. Specialty section: This article was submitted to Infectious Diseases, a section of the journal Frontiers in Microbiology Received: 01 August 2016 Accepted: 17 

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


INPUT TEXT:
Canovas1†, Mara Baldry1†, Martin S. Bojer1†, Paal S. Andersen1,2, Bengt H. Gless3, Piotr K. Grzeskowiak3, Marc Stegger2, Peter Damborg1, Christian A. Olsen3and Hanne Ingmer1* 1Department of Veterinary Disease Biology, Faculty of Health and Medical Sciences, University of Copenhagen, Frederiksberg, Denmark,2Department of Microbiology and Infection Control, Statens Serum Institut, Copenhagen, Denmark,3Center for Biopharmaceuticals and Department of Drug Design and Pharmacology, Faculty of Health a...

PROCESSED TEXT:
egger2 Peter Damborg1 Christian A Olsen3 Hanne Ingmer1 Department of Veterinary Disease Biology Faculty of Health and Medical Sciences University of Copenhagen Frederiksberg Denmark2 Department of Microbiology and Infection Control Statens Serum Institut Copenhagen Denmark3 Center for Biopharmaceuticals and Department of Drug Design and Pharmacology Faculty of Health and Medical Sciences University of Copenhagen Copenhagen Denmark Staphylococci are associated wit

...

INPUT TEXT:
features essential for activation and inhibition of an AgrC quorum sensing receptor in Staphylococcus aureus .J. Am. Chem. Soc. 135, 18436–18444. doi: 10.1021/ja3112115 Tal-Gan, Y., Stacy, D. M., Foegen, M. K., Koenig, D. W., and Blackwell, H. E. (2013b). Highly potent inhibitors of quorum sensing in Staphylococcus aureus revealed through a systematic synthetic study of the group-III autoinducing peptide. J. Am. Chem. Soc. 135, 7869–7882. doi: 10.1021/ja3112115 Tal-Gan, Y., Ivancic, M., Corniles...

PROCESSED TEXT:
aureus
J Am Chem Soc 135 18436–18444 doi 10 1021 j1312115 Tal Gan Y Stacy DM Foegen MK Koenig DW and Blackwell HE (2013b) Highly potent inhibitors of quorum sensing in Staphylococcus aureus revealed through a systematic synthetic study of the group III autoinducing peptide J Am Chem Soc 135 7869–7882 doi 10 1021 j1312115 Tal Gan Y Ivancic M Cornilescu G Y ang T and Blackwell HE (2016) Highly stable amide-bridged autoinducing peptide analogues that strongly inhibi

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


INPUT TEXT:
E., and Horswill, A. R. (2011). Peptide signaling in the staphylococci. Chem. Rev. 111, 117–151. doi: 10.1021/cr100370n Vandenesch, F. O., Projan, S. J., Kreiswirth, B., Etienne, J., and Novick, R. P. (1993). Agr-related sequences in Staphylococcus lugdunensis .FEMS Microbiol. Lett. 111, 115–122. doi: 10.1111/j.1574-6968.1993.tb06370.x Wamel, W. J. B., Rossum, G., Verhoef, J., Vandenbroucke-Grauls, C. M. J. E., and Fluit, A. C. (1998). Cloning and characterization of an accessory gene regulator ...

PROCESSED TEXT:
aphylococci. Chem. Rev. 111, 117–151. doi: 10.1021/cr100370n

Vandenesch, F. O., Projan, S. J., Kreiswirth, B., Etienne, J., and Novick, R. P. (1993). Agr-related sequences in Staphylococcus lugdunensis

Agr-related sequences in Staphylococcus epidermidis 

Wamel, W. J. B., Rossum, G., Verhoef, J., Vandenbroucke-Grauls, C. M. J. E., and Fluit, A. C. (1998). Cloning and characterization of an accessory gene regulator (agr)-like locus from Staphylococcus epidermidi

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


INPUT TEXT:
Cell. 53, 929–940. doi: 10.1016/j.molcel.2014. 02.029 Wright, J. S., Lyon, G. J., George, E. A., Muir, T. W., and Novick, R. P. (2004). Hydrophobic interactions drive ligand-receptor recognition for activation and inhibition of staphylococcal quorum sensing. Proc. Natl. Acad. Sci. U.S.A. 101, 16168–16173. doi: 10.1073/pnas.0401825101 Y ang, T., Tal-Gan, Y., Paharik, A. E., Horswill, A. R., and Blackwell, H. E. (2016). Structure-function analyses of a Staphylococcus epidermidis autoinducing pepti...

PROCESSED TEXT:
ir, T. W., and Novick, R. P. (2004) Hydrophobic interactions drive ligand-receptor recognition for activation and inhibition of staphylococcal quorum sensing Proc. Natl. Acad. Sci. U.S.A. 101, 16168–16173 Y ang, T., Tal-Gan, Y., Paharik, A. E., Horswill, A. R., and Blackwell, H. E. (2016) Structure-function analyses of a Staphylococcus epidermidis autoinducing peptide reveals motifs critical for AgrC-type receptor modulation ACS Chem. Biol. 11, 1982–1991 Zheng, J

In [16]:
#Inspect results
print(f"\nProcessing complete!")
print(f"Input file: {INPUT_FILE}")
print(f"Output file: {output_file}")
print(f"Total chunks processed: {num_chunks}")

print("\nPreview of final processed text:")
print("\nBEGINNING:")
print(processed_text[:1000])
print("\n...\n\nEND:")
print(processed_text[-1000:])


Processing complete!
Input file: extracted_text.txt
Output file: clean_extracted_text.txt
Total chunks processed: 62

Preview of final processed text:

BEGINNING:
: 08 November 2016 doi: 10.3389/fmicb.2016.01733 Edited by: Susanne Fetzner, University of Münster, Germany Reviewed by: Mattias Collin, Lund University, Sweden Shinya Watanabe, Jichi Medical University, Japan 
Correspondence: Hanne Ingmer hi@sund.ku.dk †These authors have contributed equally to this work. Specialty section: This article was submitted to Infectious Diseases, a section of the journal Frontiers in Microbiology Received: 01 August 2016 Accepted: 17 October 2016 Published: 08 November 2016 Citation: Canovas J, Baldry M, Bojer MS, Andersen PS, Gless BH, Grzeskowiak PK, Stegger M, Damborg P, Olsen CA and Ingmer H (2016) Cross-talk between Staphylococcus aureus and other Staphylococcal species via the agrQuorum Sensing System.
egger2 Peter Damborg1 Christian A Olsen3 Hanne Ingmer1 Department of Veterinary Disease B